# Categorizing Transactions Model

### Setup

In [39]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

from imblearn.over_sampling import RandomOverSampler
from xgboost import XGBClassifier
import gspread
from google.oauth2.service_account import Credentials

### Cleaning and Parse Amount and Dates

In [40]:
WORKSHEET_NAME = "All Transactions"
SHEET_ID = "1UoIaxcnFor39N_KcIwnq8JWaauPzrDWEv_S5PFcca1M"
SERVICE_ACCOUNT_FILE = "transactions-project-483516-1dcb6dfc2520.json"

# --------------------------
# Auth
# --------------------------
scopes = ["https://www.googleapis.com/auth/spreadsheets"]
creds = Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=scopes
)
client = gspread.authorize(creds)

# --------------------------
# Load sheet into DataFrame
# --------------------------
sheet = client.open_by_key(SHEET_ID)
worksheet = sheet.worksheet(WORKSHEET_NAME)

data = worksheet.get_all_records()
df = pd.DataFrame(data)
df = df[['Account', 'Amount', 'Category', 'Date', 'Description', 'Subcategory']]

# make sure categories are valid
df['Subcategory'] = df['Subcategory']
df = df[df['Subcategory'].notna()]
df = df[df['Subcategory'] != ""]

# Clean Amounts like '-$5.70', '$3.90', '$1,234.56'
df['Amount'] = (
    df['Amount']
    .astype(str)
    .str.replace('[\$,]', '', regex=True)  # remove $ and commas
    .astype(float)
)

# Parse Date
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date', 'Amount'])  # drop rows with invalid date/amount


### 3. Derived Features

In [41]:
# Numeric & Date features
df['abs_amount'] = df['Amount'].abs()
df['is_income'] = (df['Amount'] > 0).astype(int)
df['weekday'] = df['Date'].dt.weekday
df['day'] = df['Date'].dt.day
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)
df['Month'] = df["Month"] = df["Date"].dt.month_name()

### Prepare features and labels

In [42]:
feature_cols = ['Description', 'Account', 'Month', 'abs_amount', 'is_income', 'weekday', 'is_weekend', 'Category']
X = df[feature_cols]
y = df['Subcategory']

In [43]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

### Train / Test Split

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

### Column Transformer for mixed features

In [45]:
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=10000), 'Description'),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Account', 'Month', 'Category']),
        ('num', 'passthrough', ['abs_amount', 'is_income', 'weekday', 'is_weekend'])
    ]
)

### Handle Class imbalance

In [46]:
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

C:\Users\Jake Huffman\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
C:\Users\Jake Huffman\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


### Build the pipeline with XGBoost

In [47]:
model = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    use_label_encoder=False,
    n_jobs=-1,
    random_state=42
)

pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('classifier', model)
])


### Train the model

In [48]:
pipeline.fit(X_resampled, y_resampled)


C:\Users\Jake Huffman\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\xgboost\core.py:158: UserWarning: [10:45:55] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('text',
                                                  TfidfVectorizer(max_features=10000,
                                                                  min_df=2,
                                                                  ngram_range=(1,
                                                                               2)),
                                                  'Description'),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Account', 'Month',
                                                   'Category']),
                                                 ('num', 'passthrough',
                                                  ['abs_amount', 'is_income',
                                                   'weekday',
                                                   'is_weekend'])])),
                ('classifier',
                 XGBClassifier(base_score=None, boost...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=-1,
                               num_parallel_tree=None,
                               objective='multi:softprob', ...))])

### Evaluate Performance

In [49]:
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.27      0.33      0.30         9
           1       0.86      0.86      0.86        59
           2       0.33      0.50      0.40         2
           3       0.39      0.58      0.47        12
           4       0.86      0.75      0.80         8
           5       0.40      1.00      0.57         2
           6       0.90      0.86      0.88        21
           7       1.00      1.00      1.00       205
           8       0.56      0.38      0.45        13
           9       0.50      0.25      0.33         8
          10       1.00      0.88      0.94        17

    accuracy                           0.88       356
   macro avg       0.64      0.67      0.64       356
weighted avg       0.89      0.88      0.89       356



### Save the model

In [50]:
import joblib

joblib.dump(pipeline, 'subcategory_model.pkl')
joblib.dump(le, 'subcategory_label_encoder.pkl')



['subcategory_label_encoder.pkl']

### Predict for missing categories

In [52]:
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd
import joblib
import numpy as np

# --------------------------
# Load trained model + label encoder
# --------------------------
pipeline = joblib.load("subcategory_model.pkl")  # contains {'pipeline': pipeline, 'label_encoder': le}
le = joblib.load('subcategory_label_encoder.pkl')

# --------------------------
# Google Sheets auth
# --------------------------
scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file("transactions-project-483516-1dcb6dfc2520.json", scopes=scopes)
client = gspread.authorize(creds)

sheet = client.open_by_key("1UoIaxcnFor39N_KcIwnq8JWaauPzrDWEv_S5PFcca1M").worksheet("All Transactions")
rows = sheet.get_all_records()
df = pd.DataFrame(rows)

# Amount
df['Amount'] = df['Amount'].astype(str).str.replace('[\$,]', '', regex=True).astype(float)

# Date
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date', 'Amount'])

# Derived features
df['abs_amount'] = df['Amount'].abs()
df['is_income'] = (df['Amount'] > 0).astype(int)
df['weekday'] = df['Date'].dt.weekday
df['day'] = df['Date'].dt.day
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)
df['Month'] = df["Month"] = df["Date"].dt.month_name()

# --------------------------
# Preprocess sheet data
# --------------------------
df["Subcategory"] = df["Subcategory"].fillna("").str.strip()
to_predict = df[df["Subcategory"] == ""].copy()
print(f"Predicting {len(to_predict)} transactions")

# --------------------------
# Prepare model input
# --------------------------
X_pred = to_predict[['Description', 'Account', 'Month', 'abs_amount', 'is_income', 'weekday', 'is_weekend', 'Category']]

# --------------------------
# Predict categories + probabilities
# --------------------------
probs = pipeline.predict_proba(X_pred)
predicted_indices = np.argmax(probs, axis=1)
predicted_categories = le.inverse_transform(predicted_indices)
confidences = probs[np.arange(len(probs)), predicted_indices]
print(predicted_indices)
print(predicted_categories)
print(confidences)

# Pretty-print categories
def prettify_category(cat: str) -> str:
    return cat.replace("_", " ").title()

# --------------------------
# Apply confidence threshold
# --------------------------
CONF_THRESHOLD = 0.8
final_categories = [
    cat if conf >= CONF_THRESHOLD else "" 
    for cat, conf in zip(predicted_categories, confidences)
]
final_confidences = [
    conf if conf >= CONF_THRESHOLD else np.nan
    for conf in confidences
]

# --------------------------
# Update DataFrame
# --------------------------
df.loc[to_predict.index, "Subcategory"] = final_categories
df.loc[to_predict.index, "Subcat Conf"] = final_confidences
# print(final_categories)

# --------------------------
# Push updates to Google Sheets
# --------------------------
headers = sheet.row_values(1)
category_col = headers.index("Subcategory") + 1

# Add confidence column if not exist
if "Subcat Conf" in headers:
    confidence_col = headers.index("Subcat Conf") + 1
else:
    sheet.update_cell(1, len(headers)+1, "Subcat Conf")
    confidence_col = len(headers) + 1

# Convert confidence to standard Python float (or None for NaN)
final_confidences = [
    float(conf) if not pd.isna(conf) else None
    for conf in final_confidences
]

# Now push updates
updates = []
for idx in to_predict.index:
    sheet_row = idx + 2  # df index → sheet row
    updates.append({
        "range": gspread.utils.rowcol_to_a1(sheet_row, category_col),
        "values": [[df.loc[idx, "Subcategory"]]]
    })
    updates.append({
        "range": gspread.utils.rowcol_to_a1(sheet_row, confidence_col),
        "values": [[final_confidences[to_predict.index.get_loc(idx)]]]
    })

if updates:
    sheet.batch_update(updates)
    print(f"Updated {len(to_predict)} transactions with confidence threshold of {CONF_THRESHOLD*100:.0f}%")

Predicting 138 transactions
[8 0 6 6 1 3 4 6 1 1 0 8 1 8 0 6 8 1 3 8 6 6 4 3 6 3 0 4 3 3 3 8 8 0 4 4 0
 0 8 1 4 4 0 0 0 0 1 3 0 8 1 1 1 1 6 3 8 8 8 3 3 4 3 1 8 8 8 0 3 6 0 8 0 0
 0 0 0 0 8 3 0 9 9 0 9 9 9 9 6 9 9 9 9 9 4 9 9 9 9 0 9 4 9 0 9 0 3 3 0 1 3
 5 9 9 9 9 6 4 1 9 9 9 6 1 9 9 1 9 8 9 1 9 5 6 0 5 5 3]
['Other' 'Activities' 'Golf' 'Golf' 'Alcohol' 'Eating Out' 'Gambling'
 'Golf' 'Alcohol' 'Alcohol' 'Activities' 'Other' 'Alcohol' 'Other'
 'Activities' 'Golf' 'Other' 'Alcohol' 'Eating Out' 'Other' 'Golf' 'Golf'
 'Gambling' 'Eating Out' 'Golf' 'Eating Out' 'Activities' 'Gambling'
 'Eating Out' 'Eating Out' 'Eating Out' 'Other' 'Other' 'Activities'
 'Gambling' 'Gambling' 'Activities' 'Activities' 'Other' 'Alcohol'
 'Gambling' 'Gambling' 'Activities' 'Activities' 'Activities' 'Activities'
 'Alcohol' 'Eating Out' 'Activities' 'Other' 'Alcohol' 'Alcohol' 'Alcohol'
 'Alcohol' 'Golf' 'Eating Out' 'Other' 'Other' 'Other' 'Eating Out'
 'Eating Out' 'Gambling' 'Eating Out' 'Alcohol' 'Other' '

### Adding Transactions


In [83]:
import re
import pandas as pd
import csv
import numpy as np

# Chase Transactions
def process_chase_credit_card_transactions(file_name):
    # --------------------------
    # Read CSV
    # --------------------------
    df_csv = pd.read_csv(file_name)

    df_csv = df_csv.rename(columns={
        "Transaction Date": "Date",
        "Description": "Description",
        "Amount": "Amount"
    })

    # Ensure the columns exist (avoid KeyError)
    for col in ["Category", "Type"]:
        if col not in df_csv.columns:
            df_csv[col] = ""

    cat = df_csv["Category"].fillna("").astype(str).str.strip()
    typ = df_csv["Type"].fillna("").astype(str).str.strip()

    df_csv["Description"] = (
        df_csv["Description"].astype(str).str.strip()
        + np.where(cat != "", " - " + cat, "")
        + np.where(typ != "", " - " + typ, "")
    )

    df_csv["Account"] = "Chase_Credit_Card"
    df_csv = df_csv[["Date", "Description", "Amount", "Account"]]

    df_csv["Date"] = pd.to_datetime(df_csv["Date"], errors="coerce")
    df_csv["Amount"] = (
        df_csv["Amount"]
        .astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .astype(float)
    )

    df_csv = df_csv.dropna(subset=["Date", "Amount", "Description"])
    return df_csv

def process_bofa_checking_transactions(file_name):
    header_row_idx = 5  # adjust if your CSV format changes

    # Read CSV using Python engine, ignore quotes (malformed quotes handled)
    df_csv = pd.read_csv(
        file_name,
        header=header_row_idx,
        engine="python",
        dtype=str,
        quoting=csv.QUOTE_NONE,
        skip_blank_lines=True,
        on_bad_lines="skip"  # skip malformed rows instead of erroring
    )

    # Keep only the columns we care about
    df_csv = df_csv[["Date", "Description", "Amount"]]

    # Add Account column
    df_csv["Account"] = "BOFA_Checking"

    # Clean Amount column: remove $, commas, spaces, convert to float
    df_csv["Amount"] = (
        df_csv["Amount"]
        .astype(str)
        .str.replace('"', '', regex=False)   # remove extra quotes
        .str.replace(r"[\$,]", "", regex=True)  # remove $ and ,
        .str.replace(" ", "", regex=False)      # remove spaces
        .astype(float)
    )

    # Parse Dates
    df_csv["Date"] = pd.to_datetime(df_csv["Date"], errors="coerce")

    # Drop rows missing critical info
    df_csv = df_csv.dropna(subset=["Date", "Amount", "Description"])

    return df_csv

def process_bofa_credit_card_transactions(file_name):
    # --------------------------
    # Read CSV
    # --------------------------
    df_csv = pd.read_csv(file_name)

    df_csv = df_csv.rename(columns={
        "Posted Date": "Date"
    })

    df_csv["Description"] = df_csv.apply(
        lambda row: " - ".join(
            filter(None, [
                str(row["Reference Number"]) if not pd.isna(row["Reference Number"]) else "",
                str(row["Payee"]) if not pd.isna(row["Payee"]) else "",
                str(row["Address"]) if not pd.isna(row["Address"]) else ""
            ])
        ),
        axis=1
    )

    df_csv["Account"] = "BOFA_Credit_Card"
    df_csv = df_csv[["Date", "Description", "Amount", "Account"]]

    df_csv["Date"] = pd.to_datetime(df_csv["Date"], errors="coerce")
    df_csv["Amount"] = (
        df_csv["Amount"]
        .astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .astype(float)
    )

    df_csv = df_csv.dropna(subset=["Date", "Amount", "Description"])
    return df_csv

def parse_venmo_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, skiprows=2)

    df = df[
        df["ID"].notna() &
        df["Datetime"].notna() &
        df["Amount (total)"].notna()
    ].copy()

    df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")

    return df.reset_index(drop=True)

def parse_venmo_amount(s):
    """
    Convert a Venmo amount string to float.
    Handles:
      - '- $13.00', '- 13.00', '-\xa013.00' (NBSP)
      - '+ $170.00'
      - '$1,727.09'
    """
    s = str(s).strip()  # remove leading/trailing whitespace
    if s == "":
        return 0.0

    # Detect sign
    sign = 1
    if s.startswith("-"):
        sign = -1
    elif s.startswith("+"):
        sign = 1

    # Remove everything except digits and dot
    num = re.sub(r"[^\d.]", "", s)

    if num == "":
        return 0.0

    return sign * float(num)

# Come back to this
def process_venmo_transactions(file_name):
    # --------------------------
    # Read CSV
    # --------------------------
    df = parse_venmo_csv(file_name)

    # --------------------------
    # Rename columns
    # --------------------------
    df = df.rename(columns={
        "Datetime": "Date"
    })

    # --------------------------
    # Normalize Amount
    # --------------------------
    df["Amount"] = df["Amount (total)"].apply(parse_venmo_amount)


    # --------------------------
    # Build Description (vectorized)
    # --------------------------
    df["Description"] = (
        "Type " + df["Type"].fillna("")
        + ", From: " + df["From"].fillna("")
        + ", To: " + df["To"].fillna("")
        + ", Note: " + df["Note"].fillna("")
    )

    # --------------------------
    # Clean Date
    # --------------------------
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    df["Account"] = "Venmo"

    # --------------------------
    # Final column order
    # --------------------------
    return df[["Date", "Description", "Amount", "Account"]]

def process_citi_bank_credit_card_transactions(file_name):
    # --------------------------
    # Read CSV
    # --------------------------
    df = pd.read_csv(file_name)

    df["Account"] = "Citi_Credit_Card"

    df["Debit"] = (
        df["Debit"]
        .astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .replace("", np.nan)
        .astype(float)
    )

    df["Credit"] = (
        df["Credit"]
        .astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .replace("", np.nan)
        .astype(float)
    )

    # Use Debit if present, otherwise Credit
    df["Amount"] = df["Debit"].fillna(df["Credit"])

    # Reverse sign
    df["Amount"] = -df["Amount"]

    df = df[["Date", "Description", "Amount", "Account"]]

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Amount"] = (
        df["Amount"]
        .astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .astype(float)
    )

    df = df.dropna(subset=["Date", "Amount", "Description"])
    return df

def parse_paystub(csv_path: str) -> dict[str, pd.DataFrame]:
    """
    Parse a vertically-stacked paystub CSV into logical DataFrames.

    Returns:
        dict[str, pd.DataFrame]
        Keys include:
        - company_info
        - payslip_info
        - totals
        - earnings
        - taxes
        - post_tax_deductions
        - employer_benefits
        - subject_wages
        - payment_info
    """

    # --------------------------
    # Read raw CSV (no headers)
    # --------------------------
    df = pd.read_excel(csv_path, header=None)
    df = df.fillna("")

    # --------------------------
    # Section header labels
    # --------------------------
    SECTION_TITLES = {
        "Company Information": "company_info",
        "Payslip Information": "payslip_info",
        "Current and YTD Totals": "totals",
        "Earnings": "earnings",
        "Employee Taxes Withheld": "taxes",
        "Post-Tax Deductions": "post_tax_deductions",
        "Employer Paid Benefits": "employer_benefits",
        "Subject Wages": "subject_wages",
        "Payment Information": "payment_info",
    }

    # --------------------------
    # Locate section start rows
    # --------------------------
    section_rows = {}

    for idx, value in df[0].items():
        if value in SECTION_TITLES:
            section_rows[value] = idx

    # Preserve order
    sorted_sections = sorted(
        section_rows.items(),
        key=lambda x: x[1]
    )

    results = {}

    # --------------------------
    # Extract each section
    # --------------------------
    for i, (title, start_row) in enumerate(sorted_sections):
        key = SECTION_TITLES[title]

        # Header is always next row
        header_row = start_row + 1
        data_start = header_row + 1

        # End is next section start or EOF
        if i + 1 < len(sorted_sections):
            end_row = sorted_sections[i + 1][1]
        else:
            end_row = len(df)

        table = df.iloc[data_start:end_row].copy()
        headers = df.iloc[header_row].tolist()

        # Trim empty columns
        valid_cols = [i for i, h in enumerate(headers) if h != ""]
        headers = [headers[i] for i in valid_cols]
        table = table.iloc[:, valid_cols]

        table.columns = headers
        table = table.reset_index(drop=True)

        # Drop fully empty rows
        table = table[
            table.apply(lambda r: any(str(v).strip() for v in r), axis=1)
        ]

        results[key] = table

    # --------------------------
    # Numeric cleanup
    # --------------------------
    NUMERIC_COL_HINTS = [
        "Amount", "YTD", "Hours", "Rate", "Gross", "Net"
    ]

    for df_name, df_table in results.items():
        for col in df_table.columns:
            if any(hint in col for hint in NUMERIC_COL_HINTS):
                df_table[col] = (
                    df_table[col]
                    .astype(str)
                    .str.replace(",", "")
                    .replace("", None)
                )
                df_table[col] = pd.to_numeric(df_table[col], errors="coerce")

    return results

def extract_pay_date(earnings_df: pd.DataFrame) -> pd.Timestamp:
    """
    Extracts the LAST date from the 'Dates' column in Earnings.
    Example: '12/07/2025 - 12/20/2025' → 2025-12-20
    """
    dates_col = earnings_df["Dates"].dropna().astype(str)

    # Find first row with a date range
    for val in dates_col:
        if "-" in val:
            end_date = val.split("-")[-1].strip()
            return pd.to_datetime(end_date)

    raise ValueError("Could not extract pay date from Earnings table")

def paystub_to_transactions(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    pay_date = extract_pay_date(tables["earnings"])
    transactions = []

    def add_txn(desc, amount):
        transactions.append({
            "Date": pay_date,
            "Description": desc,
            "Amount": round(float(amount), 2)
        })

    def add_rows(
        df,
        sign=1,
        prefix="",
        allowed_descriptions=None,
        excluded_descriptions=None
    ):
        for _, row in df.iterrows():
            desc = str(row.get("Description", "")).strip()
            amount = row.get("Amount")

            if not desc or pd.isna(amount) or amount == 0:
                continue

            if allowed_descriptions and desc not in allowed_descriptions:
                continue

            if excluded_descriptions and desc in excluded_descriptions:
                continue

            add_txn(
                f"{prefix}{desc}",
                sign * amount
            )

    # --------------------------
    # Earnings (+) — EXCLUDE TERM LIFE
    # --------------------------
    add_rows(
        tables["earnings"],
        sign=+1,
        prefix="Paystub Income: ",
        excluded_descriptions={
            "Imputed Income - Group Term Life"
        }
    )

    # --------------------------
    # Employer Benefits (+ + OFFSET)
    # --------------------------
    BENEFITS_ALLOWLIST = {
        "401(k) Match",
        "Health Savings Account"
    }

    if "employer_benefits" in tables:
        for _, row in tables["employer_benefits"].iterrows():
            desc = str(row.get("Description", "")).strip()
            amount = row.get("Amount")

            if desc not in BENEFITS_ALLOWLIST or pd.isna(amount) or amount == 0:
                continue

            # Positive asset contribution
            add_txn(
                f"Employer Benefit: {desc}",
                +amount
            )

            # Negative offset (cash-neutral)
            add_txn(
                f"Employer Benefit Offset: {desc}",
                -amount
            )

    # --------------------------
    # Taxes (-)
    # --------------------------
    add_rows(
        tables["taxes"],
        sign=-1,
        prefix="Tax: "
    )

    # --------------------------
    # Post-tax deductions (-)
    # --------------------------
    if "post_tax_deductions" in tables:
        add_rows(
            tables["post_tax_deductions"],
            sign=-1,
            prefix="Post-Tax Deduction: "
        )

    return pd.DataFrame(transactions)

def process_paystub_transactions(file_name):
    # --------------------------
    # Read CSV
    # --------------------------
    tables = parse_paystub(file_name)
    df_txns = paystub_to_transactions(tables)
    df_txns["Account"] = "Paystub"

    assert round(
        paystub_to_transactions(tables)["Amount"].sum(), 2
    ) == round(
        tables["totals"]
            .query("`Balance Period` == 'Current'")["Net Pay"]
            .iloc[0],
        2
    ), "Paystub parsing imbalanced."

    return df_txns


In [88]:
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
import joblib
import numpy as np

# --------------------------
# Configuration
# --------------------------
CONF_THRESHOLD = 0.8
SHEET_NAME = "All Transactions"
SPREADSHEET_KEY = "1UoIaxcnFor39N_KcIwnq8JWaauPzrDWEv_S5PFcca1M"

def process_statement(account, file_name):

    # --------------------------
    # Google Sheets auth
    # --------------------------
    scopes = [
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive"
    ]

    creds = Credentials.from_service_account_file(
        "transactions-project-483516-1dcb6dfc2520.json",
        scopes=scopes
    )
    client = gspread.authorize(creds)
    sheet = client.open_by_key(SPREADSHEET_KEY).worksheet(SHEET_NAME)

    # --------------------------
    # Load model + encoder
    # --------------------------
    pipeline = joblib.load("transaction_model.pkl")
    le = joblib.load("label_encoder.pkl")

    if account == "BOFA_Checking":
        df_csv = process_bofa_checking_transactions(file_name)
    elif account == "BOFA_Credit_Card":
        df_csv = process_bofa_credit_card_transactions(file_name)
    elif account == "Chase_Credit_Card":
        df_csv = process_chase_credit_card_transactions(file_name)
    elif account == "Citi_Credit_Card":
        df_csv = process_citi_bank_credit_card_transactions(file_name)
    elif account == "Venmo":
        df_csv = process_venmo_transactions(file_name)
    elif account == "Paystub":
        df_csv = process_paystub_transactions(file_name)
    else:
        df_csv = pd.DataFrame()
        print("Invalid ACCOUNT name")

    # --------------------------
    # Fetch existing transactions
    # --------------------------
    existing_rows = sheet.get_all_records()
    df_existing = pd.DataFrame(existing_rows)

    # --------------------------
    # Duplicate detection (NON-DESTRUCTIVE)
    # --------------------------
    def build_dup_key(df):
        # Normalize date
        date_part = (
            pd.to_datetime(df["Date"], errors="coerce")
            .dt.strftime("%Y-%m-%d")
        )

        # Normalize description (ONLY for key)
        desc_part = (
            df["Description"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        # Normalize amount (handle $, commas, blanks)
        amount_part = (
            df["Amount"]
            .astype(str)
            .str.replace(r"[\$,]", "", regex=True)
            .str.strip()
        )

        amount_part = pd.to_numeric(amount_part, errors="coerce").round(2)

        account_part = df["Account"]

        return (
            date_part
            + "|"
            + desc_part
            + "|"
            + amount_part.astype(str)
            + "|"
            + account_part
        )


    df_csv["_dup_key"] = build_dup_key(df_csv)
    df_existing["_dup_key"] = build_dup_key(df_existing)

    df_new = df_csv[~df_csv["_dup_key"].isin(df_existing["_dup_key"])].copy()

    print(f"Found {len(df_new)} new transactions to add.")

    if df_new.empty:
        print("No new transactions to process.")
    else:

        # --------------------------
        # Feature engineering
        # --------------------------
        df_new["abs_amount"] = df_new["Amount"].abs()
        df_new["is_income"] = (df_new["Amount"] > 0).astype(int)
        df_new["weekday"] = df_new["Date"].dt.weekday
        df_new["is_weekend"] = df_new["weekday"].isin([5, 6]).astype(int)
        df_new["Month"] = df_new["Date"].dt.month.astype(str)

        X_pred = df_new[
            ["Description", "Account", "Month",
            "abs_amount", "is_income", "weekday", "is_weekend"]
        ].copy()

        X_pred["Description"] = X_pred["Description"].astype(str)
        X_pred["Account"] = X_pred["Account"].astype(str)
        X_pred["abs_amount"] = X_pred["abs_amount"].astype(float)
        X_pred["is_income"] = X_pred["is_income"].astype(int)
        X_pred["weekday"] = X_pred["weekday"].astype(int)
        X_pred["is_weekend"] = X_pred["is_weekend"].astype(int)

        # --------------------------
        # Predict categories
        # --------------------------
        try:
            probs = pipeline.predict_proba(X_pred)
            pred_idx = np.argmax(probs, axis=1)
            pred_categories = le.inverse_transform(pred_idx)
            confidences = probs[np.arange(len(probs)), pred_idx]

            df_new["Category"] = [
                cat if conf >= CONF_THRESHOLD else ""
                for cat, conf in zip(pred_categories, confidences)
            ]
            df_new["Confidence"] = confidences.round(3)
        except:
            print("No empty categories to predict.")

        # --------------------------
        # Append to Google Sheets (header-aware)
        # --------------------------
        headers = sheet.row_values(1)
        col_map = {h: i for i, h in enumerate(headers)}

        required_headers = {
            "Date", "Description", "Amount",
            "Account", "Category", "Confidence"
        }

        missing = required_headers - set(headers)
        if missing:
            raise ValueError(f"Missing required sheet columns: {missing}")

        rows_to_append = []

        for _, row in df_new.iterrows():
            sheet_row = [""] * len(headers)

            sheet_row[col_map["Date"]] = row["Date"].strftime("%Y-%m-%d")
            sheet_row[col_map["Description"]] = row["Description"]  # UNMODIFIED
            sheet_row[col_map["Amount"]] = row["Amount"]
            sheet_row[col_map["Account"]] = row["Account"]
            sheet_row[col_map["Category"]] = row["Category"]
            sheet_row[col_map["Confidence"]] = row["Confidence"]

            rows_to_append.append(sheet_row)

        sheet.append_rows(rows_to_append, value_input_option="USER_ENTERED")
        print(f"Appended {len(rows_to_append)} new transactions.")


### Process all statements

In [92]:
import os

# Folder containing your CSV statements
STATEMENTS_DIR = "statements"

# Loop through all files in the folder
for file_name in os.listdir(STATEMENTS_DIR):
    file_path = os.path.join(STATEMENTS_DIR, file_name)
    
    # Skip directories, only process files
    if not os.path.isfile(file_path):
        continue

    # Determine account based on file name
    account = None
    fname_lower = file_name.lower()  # make it case-insensitive

    if "venmo" in fname_lower:
        account = "Venmo"
    if "_9389" in fname_lower:
        account = "BOFA_Credit_Card"
    elif "jake_huffman" in fname_lower:
        account = "Paystub"
    elif "chase" in fname_lower:
        account = "Chase_Credit_Card"
    elif "stmt" in fname_lower:
        account = "BOFA_Checking"
    elif "date range" in fname_lower:
        account = "Citi_Credit_Card"

    # Process the statement if account identified
    if account:
        print(f"Processing {file_name} as {account} statement...")
        process_statement(account, file_path)
    else:
        print(f"Could not categorize file: {file_name}")


Processing Chase8955_Activity20251211_20260101_20260101.CSV as Chase_Credit_Card statement...
Found 0 new transactions to add.
No new transactions to process.
Processing currentTransaction_9389 (14).csv as BOFA_Credit_Card statement...
Found 0 new transactions to add.
No new transactions to process.
Processing Date range (1).CSV as Citi_Credit_Card statement...
Found 0 new transactions to add.
No new transactions to process.
Processing Jake_Huffman_(100502)_12_20_2025_(Regular)_-_Complete (1).xlsx as Paystub statement...


C:\Users\Jake Huffman\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Found 0 new transactions to add.
No new transactions to process.
Processing stmt (24).csv as BOFA_Checking statement...
Found 0 new transactions to add.
No new transactions to process.
Processing VenmoStatement_December_2025 (2).csv as Venmo statement...
Found 0 new transactions to add.
No new transactions to process.


In [103]:
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials

# --------------------------
# Config
# --------------------------

WORKSHEET_NAME = "All Transactions"
SHEET_ID = "1UoIaxcnFor39N_KcIwnq8JWaauPzrDWEv_S5PFcca1M"
SERVICE_ACCOUNT_FILE = "transactions-project-483516-1dcb6dfc2520.json"

# --------------------------
# Auth
# --------------------------
scopes = ["https://www.googleapis.com/auth/spreadsheets"]
creds = Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=scopes
)
client = gspread.authorize(creds)

# --------------------------
# Load sheet into DataFrame
# --------------------------
sheet = client.open_by_key(SHEET_ID)
worksheet = sheet.worksheet(WORKSHEET_NAME)

data = worksheet.get_all_records()
df = pd.DataFrame(data)

# --------------------------
# Clean & convert Amount
# --------------------------
df["Amount_num"] = (
    df["Amount"]
    .str.replace(r"[\$,]", "", regex=True)
    .astype(float)
)

# --------------------------
# Build mask
# --------------------------
mask = (
    (df["Category"] == "Discretionary") &
    (df["Subcategory"] == "") &
    (df["Amount_num"].abs() > 0) &
    (df["Description"].str.contains("Apotheca", case=False, na=False))
)

# --------------------------
# Update Subcategory
# --------------------------
df.loc[mask, "Subcategory"] = "Drugs"

# --------------------------
# Push back to Google Sheet
# --------------------------
worksheet.update(
    [df.drop(columns=["Amount_num"]).columns.tolist()]
    + df.drop(columns=["Amount_num"]).values.tolist()
)

print(f"Updated {mask.sum()} rows")

Updated 0 rows
